# 05.7 — Batch-norm state synchronization

Most of a model's state is its **weights**, updated by gradients ([02.5](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb)). But normalization layers (BatchNorm, InstanceNorm) carry a *second* kind of state — **running statistics** (mean, variance) — that update **outside** the gradient path, as a side effect of each forward pass in training mode. This buffers-vs-parameters distinction is subtle, matters for checkpoints and eval, and is what MATLAB's `cgg_updateState` handles explicitly. This notebook closes Module 05 on it.

This notebook covers:

* Parameters vs buffers — two kinds of module state.
* How BatchNorm's running stats update in the forward pass, not backward.
* Why `eval()` freezes them (and why forgetting it corrupts inference).
* What `state_dict` carries, and MATLAB's explicit `updateState`.

**Prerequisites:** [02.6 nn.Module](../02_numpy_and_pytorch_basics/02.6_nn_module_vs_layergraph.ipynb), [05.1 the loop](05.1_the_custom_training_loop.ipynb).

## Section 1 — What MATLAB does

MATLAB's custom loop updates the network *state* separately from its learnables, via `cgg_updateState`:

```matlab
[loss, grads, state] = dlfeval(@modelGradients, net, X, T);
net.State = state;                                  % ← update running stats EXPLICITLY
[net, ...] = adamupdate(net, grads, ...);           % update LEARNABLES via gradients
```

`net.State` holds batch-norm's running mean/variance. Note it's updated from the forward pass's returned `state`, NOT from gradients — MATLAB makes this two-track update visible. PyTorch does the same thing, but *automatically* (inside the BatchNorm layer's forward), which is convenient but easier to get wrong.

## Section 2 — The Python concepts you need

### 2.1 — Parameters vs buffers

An `nn.Module` has two kinds of persistent state:

| | **Parameters** | **Buffers** |
|---|---|---|
| examples | weights, biases | BatchNorm running mean/var |
| updated by | gradients (`optimizer.step()`) | forward pass side-effects (training mode) |
| in `parameters()`? | yes | **no** |
| in `state_dict()`? | yes | **yes** |
| `requires_grad`? | typically yes | no |

Both persist and both are saved — but only parameters are trained by the optimizer. Buffers are the "other" state. A BatchNorm layer has both: learnable scale/shift (parameters) AND running mean/var (buffers):

In [ ]:
import torch
from torch import nn

bn = nn.BatchNorm1d(4)
print("parameters (gradient-trained):", [n for n, _ in bn.named_parameters()])
print("buffers    (forward-updated): ", [n for n, _ in bn.named_buffers()])

`weight`/`bias` are the affine parameters the optimizer moves; `running_mean`/`running_var` are buffers the forward pass maintains; `num_batches_tracked` counts updates. All five are in `state_dict` (so checkpoints capture them), but only the first two are in `parameters()` (so only they get gradients).

### 2.2 — Running stats update in the FORWARD pass

This is the counterintuitive part. In **training mode**, every forward call updates the running statistics toward the current batch's mean/var — *before* any backward, and independent of gradients. Watch the buffer move with forwards alone, no `backward()` anywhere:

In [ ]:
bn = nn.BatchNorm1d(4)
bn.train()                                  # training mode
print("initial running_mean:", bn.running_mean.tolist())

for step in range(3):
    x = torch.randn(16, 4) + 5.0            # data centered at 5
    _ = bn(x)                                # FORWARD ONLY — no backward, no optimizer
    print(f"after forward {step}: running_mean ≈ {[round(v, 3) for v in bn.running_mean.tolist()]}")
print("→ the buffer drifted toward the data's mean (5), driven by forwards alone.")

No `backward()`, no `optimizer.step()`, yet the buffer changed — because BatchNorm's forward, in training mode, does a running update `running_mean ← (1−momentum)·running_mean + momentum·batch_mean`. This is exactly MATLAB's `net.State = state` assignment, just hidden inside the layer. The consequence: **a forward pass mutates the model** (its buffers), which is why validation must be careful.

### 2.3 — `eval()` freezes the buffers

In **eval mode**, BatchNorm stops updating the running stats and *uses* them instead of the batch's own statistics. This is why [05.1 §2.2](05.1_the_custom_training_loop.ipynb)'s `model.eval()` is mandatory around validation — without it, validation forwards would (a) corrupt the running stats with val data, and (b) normalize using the val batch instead of the learned training statistics:

In [ ]:
bn = nn.BatchNorm1d(4); bn.train()
for _ in range(10): bn(torch.randn(32, 4) + 3.0)     # train the stats toward mean 3
trained_mean = bn.running_mean.clone()

bn.eval()                                             # freeze
for _ in range(5): bn(torch.randn(32, 4) + 99.0)     # feed WILD data...
print("running_mean changed by eval-mode forwards?", not torch.equal(trained_mean, bn.running_mean))
print("(False = frozen: eval mode ignores the new data's stats — as it must)")

In eval mode the buffers are inert — the wild mean-99 data didn't budge them. That's the correct behavior: at inference you normalize by what the model learned during training, not by whatever batch happens to be in front of it. Forget `eval()` and your validation both corrupts the model and mis-normalizes — the silent metric-degradation bug from [02.6 §5](../02_numpy_and_pytorch_basics/02.6_nn_module_vs_layergraph.ipynb).

### 2.4 — Buffers travel in `state_dict` (so checkpoints are complete)

Because the running stats are essential to reproduce the model's behavior, they must survive a save/load — and they do, via `state_dict` ([05.5](05.5_checkpoint_resume_state_machine.ipynb)). A checkpoint that saved only `parameters()` would lose them and produce different outputs on reload:

In [ ]:
import tempfile
from pathlib import Path

bn = nn.BatchNorm1d(4); bn.train()
for _ in range(10): bn(torch.randn(32, 4) + 7.0)     # accumulate distinctive stats

d = Path(tempfile.mkdtemp())
torch.save(bn.state_dict(), d / "bn.pt")             # state_dict INCLUDES buffers
bn2 = nn.BatchNorm1d(4)
bn2.load_state_dict(torch.load(d / "bn.pt", weights_only=True))
print("running_mean survived save/load?", torch.equal(bn.running_mean, bn2.running_mean))
print("(this is why 05.5's state_dict checkpoint is COMPLETE — buffers included)")

## Section 3 — The `neural_data_decoding` implementation

This project's encoders can use **InstanceNorm** (`want_normalization: 'Instance'` on the conv encoders, [04.4](../04_architecture/04.4_convolutional_backbones.ipynb)) or **LayerNorm** ([04.2](../04_architecture/04.2_building_a_simple_encoder.ipynb)). LayerNorm has no running buffers (it normalizes per-sample, so nothing to track across batches) — which sidesteps the whole sync issue. Where normalization with buffers IS used, the `train()`/`eval()` discipline from [05.1](05.1_the_custom_training_loop.ipynb) handles the sync automatically. Confirm which norms carry buffers:

In [ ]:
for norm_name, norm in [("LayerNorm", nn.LayerNorm(8)),
                        ("BatchNorm1d", nn.BatchNorm1d(8)),
                        ("InstanceNorm1d(track_running_stats=True)",
                         nn.InstanceNorm1d(8, track_running_stats=True))]:
    buffers = [n for n, _ in norm.named_buffers()]
    print(f"  {norm_name:42} buffers: {buffers or '(none — no cross-batch state)'}")

Things to spot:

* **LayerNorm has no buffers** — it computes statistics per-sample at every forward, in both train and eval mode, so there's no running state to synchronize. This is one reason it's a convenient default: no `eval()`-forgetting footgun.
* **BatchNorm/InstanceNorm-with-tracking DO carry buffers** — and rely on the `train()`/`eval()` mode flag ([02.6 §2.5](../02_numpy_and_pytorch_basics/02.6_nn_module_vs_layergraph.ipynb)) to know whether to update them.
* The production loop's `model.train()` / `model.eval()` around each epoch/validation ([05.1 §2.2](05.1_the_custom_training_loop.ipynb)) is the automatic equivalent of MATLAB's explicit `net.State = state` — PyTorch just does the buffer bookkeeping inside the layer, so the loop only has to flip the mode.

## Section 4 — Hands-on exercises

### Exercise 4.1 — parameters vs buffers count

For a `BatchNorm1d(10)`, list which state entries are parameters (trained) and which are buffers (forward-updated), and confirm only the parameters appear in `.parameters()`.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
bn = nn.BatchNorm1d(10)
params = {n for n, _ in bn.named_parameters()}
buffers = {n for n, _ in bn.named_buffers()}
state = set(bn.state_dict().keys())
print("parameters (in .parameters()):", sorted(params))
print("buffers    (NOT in .parameters()):", sorted(buffers))
print("state_dict carries BOTH:", state == params | buffers)

### Exercise 4.2 — the forgot-eval() corruption

Show, concretely, that running validation forwards in TRAIN mode corrupts a BatchNorm's running stats, while EVAL mode leaves them intact.

In [ ]:
# Reveal:
def stats_after_val(mode):
    bn = nn.BatchNorm1d(4); bn.train()
    for _ in range(10): bn(torch.randn(32, 4) + 2.0)    # 'training'
    baseline = bn.running_mean.clone()
    getattr(bn, mode)()                                  # 'train' or 'eval' for validation
    for _ in range(5): bn(torch.randn(32, 4) + 50.0)     # 'validation' with different data
    return torch.equal(baseline, bn.running_mean)

print("val in eval() mode → stats intact?  ", stats_after_val("eval"))
print("val in train() mode → stats intact? ", stats_after_val("train"), "← CORRUPTED by val data")

## Section 5 — Common errors

### Validation metrics worse than training / unstable

Forgot `model.eval()` — BatchNorm normalizes by the val batch and its running stats get corrupted (Exercise 4.2). The [05.1 §2.2](05.1_the_custom_training_loop.ipynb) `eval()`/`train()` dance prevents it.

### Model behaves differently after reload despite same weights

The checkpoint saved `parameters()` only, not `state_dict()` — so BatchNorm buffers were lost (§2.4). Always save/load `state_dict`, which includes buffers.

### Buffers don't update during training

You're in `eval()` mode when you meant `train()`. Running stats only update in training mode (§2.2). Flip back after validation.

### BatchNorm with a batch size of 1 errors or misbehaves

BatchNorm needs >1 sample per batch to compute a batch statistic. With gradient accumulation ([05.2](05.2_gradient_accumulation.ipynb)), a micro-batch of 1 hits this — a reason to prefer LayerNorm/InstanceNorm (no batch dependency) for small micro-batches.

### `state_dict` keys mismatch on load after changing normalization

Switching LayerNorm ↔ BatchNorm changes the buffer keys, so old checkpoints won't load ([02.6 §5](../02_numpy_and_pytorch_basics/02.6_nn_module_vs_layergraph.ipynb)). The normalization choice is part of the architecture contract.

## Section 6 — Further reading

- [BatchNorm docs](https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm1d.html) — the running-stats update rule and `track_running_stats`.
- [register_buffer docs](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer) — how non-parameter state is declared.
- [Batch Normalization paper](https://arxiv.org/abs/1502.03167) — the original, for the train/eval distinction.

**Module 05 is complete.** You've walked the training loop end to end — the core loop, accumulation, clipping, LR control, checkpoint/resume, the two-stage lifecycle, and now normalization state. Module 06 (loss orchestration) is where the multi-objective loss these loops minimize gets assembled. The [curriculum map](../README.md) tracks what's next.